<a href="https://colab.research.google.com/github/yanchenliu-cxk/ESM/blob/main/PyTorch_Geometric_GNN_3D_DON.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import torch
from rdkit import Chem
from torch_geometric.data import Data
from torch_geometric.nn import GCNConv, global_mean_pool

# 1. 图结构提取与张量转换 (Featurization)
def smiles_to_graph(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None

    # 提取节点特征：此处以原子序数和度数作为示例，实际可加入更复杂的立体特征
    node_features = []
    for atom in mol.GetAtoms():
        node_features.append([atom.GetAtomicNum(), atom.GetDegree()])
    x = torch.tensor(node_features, dtype=torch.float)

    # 提取边特征与构建邻接矩阵
    edge_indices = []
    edge_attrs = []
    for bond in mol.GetBonds():
        i = bond.GetBeginAtomIdx()
        j = bond.GetEndAtomIdx()
        bond_type = bond.GetBondTypeAsDouble()
        # 无向图双向连接
        edge_indices.extend([[i, j], [j, i]])
        edge_attrs.extend([[bond_type], [bond_type]])

    edge_index = torch.tensor(edge_indices, dtype=torch.long).t().contiguous()
    edge_attr = torch.tensor(edge_attrs, dtype=torch.float)

    return Data(x=x, edge_index=edge_index, edge_attr=edge_attr)

# 2. 定义处理反应的 GNN 网络架构
class ReactionGNN(torch.nn.Module):
    def __init__(self, node_dim, hidden_dim, output_dim):
        super(ReactionGNN, self).__init__()
        # 定义图卷积层
        self.conv1 = GCNConv(node_dim, hidden_dim)
        self.conv2 = GCNConv(hidden_dim, hidden_dim)
        # 全连接层映射到目标维度 (如 1280 维)
        self.fc = torch.nn.Linear(hidden_dim, output_dim)

    def forward_molecule(self, data):
        # 消息传递机制处理单一分子
        x, edge_index, batch = data.x, data.edge_index, data.batch
        x = torch.relu(self.conv1(x, edge_index))
        x = torch.relu(self.conv2(x, edge_index))

        if batch is None:
            batch = torch.zeros(x.size(0), dtype=torch.long, device=x.device)

        # 全局池化，将原子节点特征聚合成分子级别的向量
        x = global_mean_pool(x, batch)
        return self.fc(x)

    def forward_reaction(self, reactant_graph, product_graph):
        # 核心逻辑：分别生成底物和产物的嵌入向量
        emb_reactant = self.forward_molecule(reactant_graph)
        emb_product = self.forward_molecule(product_graph)

        # 计算差异向量 (Difference Representation) 来表征发生的代谢反应
        reaction_embedding = emb_product - emb_reactant
        return reaction_embedding

# 3. 运行模型提取 DON -> 3-epi-DON 的反应特征
don_smiles = "CC1=C[C@@H]2[C@]([C@@H](C1=O)O)([C@]3(C[C@H]([C@H]([C@@]34CO4)O2)O)C)CO"
epi_don_smiles = "CC1=C[C@@H]2[C@]([C@@H](C1=O)O)([C@]3(C[C@H]([C@@H]([C@@]34CO4)O2)O)C)CO"

don_graph = smiles_to_graph(don_smiles)
epi_don_graph = smiles_to_graph(epi_don_smiles)

# 实例化网络：输入特征维度为2，隐藏层64，输出 1280 维以对齐 ESM-2
gnn_model = ReactionGNN(node_dim=2, hidden_dim=64, output_dim=1280)
gnn_model.eval()

with torch.no_grad():
    # 输入底物图和产物图，直接获取描述该反应的 1280 维向量
    reaction_vector = gnn_model.forward_reaction(don_graph, epi_don_graph)

print(f"提取完成！该反应的全局特征向量维度为: {reaction_vector.shape}")

提取完成！该反应的全局特征向量维度为: torch.Size([1, 1280])


In [3]:
import numpy as np

# 确保张量已转移到 CPU 内存，并转换为 NumPy 数组格式
reaction_vector_np = reaction_vector.cpu().numpy()

# 将反应特征向量保存为.npy 文件
np.save("don_gnn_reaction_vector.npy", reaction_vector_np)
print("反应特征向量已成功保存为 don_gnn_reaction_vector.npy")

反应特征向量已成功保存为 don_gnn_reaction_vector.npy
